# A1.3 · Authorization models that make bad grants impossible

**Function A — Security Architecture & Platform → The Security Architect**  ·  *Security of AI*

Builds on **[A1.2 · Designing the agent control plane](https://spbreed.github.io/cyber-commons/lessons/A1.2.html)**.

| | |
|---|---|
| Open-source tooling | OpenFGA, OPA |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


There are three ways to stop a bad permission grant, and they are not equally
good.

1. **Review it.** A human reads the request and says no. Works until Friday
   afternoon, or until the requester is persuasive, or until the reviewer is on
   holiday.
2. **Detect it.** You find the bad grant afterwards, in an access review. Better
   than nothing; the window between grant and detection is your exposure.
3. **Make it unrepresentable.** The system cannot express the grant at all. The
   request fails at the point of issue, with no human in the loop.

Only the third one scales, and the mechanism is a **ceiling**: a declared upper
bound on what each identity may *ever* hold, enforced by the thing that issues
credentials rather than by the thing that reviews them.

This matters more for agents than for people because agents get their
permissions programmatically, at machine speed, often from other agents. A
review step in that path is not a control; it is a bottleneck that will be
removed.

## 2 · Demo — ceilings, and what they refuse

Real scopes from a real deployment: a CI/CD estate with a human engineer and three service identities.

In [ ]:
# The ceiling: what each identity may hold AT MOST, whoever asks, forever.
CEILINGS = {
    "dana@corp":       {"repo:read", "repo:write", "deploy:staging", "deploy:prod",
                        "secrets:read"},
    "ci-builder":      {"repo:read", "artifact:write"},
    "deploy-bot":      {"artifact:read", "deploy:staging"},
    "triage-agent":    {"repo:read", "finding:comment"},
}

class GrantRefused(Exception):
    """Refusing is the feature, not an error path."""

def grant(identity, scopes):
    ceiling = CEILINGS.get(identity, set())
    excess = set(scopes) - ceiling
    if excess:
        raise GrantRefused(
            f"{identity} may never hold {sorted(excess)} "
            f"(ceiling: {sorted(ceiling)})")
    return set(scopes)

requests = [
    ("ci-builder",   {"repo:read", "artifact:write"},  "the normal build grant"),
    ("deploy-bot",   {"deploy:staging"},               "staging deploy"),
    ("deploy-bot",   {"deploy:prod"},                  "'just for the hotfix'"),
    ("triage-agent", {"repo:write"},                   "'so it can fix what it finds'"),
    ("ci-builder",   {"secrets:read"},                 "'the build needs a token'"),
]
for identity, scopes, why in requests:
    try:
        grant(identity, scopes)
        print(f"GRANTED  {identity:14s} {sorted(scopes)}   — {why}")
    except GrantRefused as e:
        print(f"REFUSED  {identity:14s} {sorted(scopes)}   — {why}")
        print(f"         {e}")

## 3 · Where it breaks

Three of those five requests are ones a real engineer would file with a straight face, and a reviewer would probably approve at least two. "Just for the hotfix" is how `deploy-bot` ends up with permanent production rights.

But a ceiling has a hole in it, and it is worth seeing rather than being told about: **the ceiling constrains a single identity, not a chain of them.** If `triage-agent` cannot hold `repo:write`, but it can ask `ci-builder` to act for it, the ceiling has been walked around without ever being violated.

In [ ]:
# Each individual grant is legal. The composition is not.
def call_chain(chain):
    print(" → ".join(chain))
    held = set()
    for identity in chain:
        held |= CEILINGS.get(identity, set())
    return held

reachable = call_chain(["triage-agent", "ci-builder", "deploy-bot"])
print("scopes reachable through the chain:", sorted(reachable))
print("triage-agent's own ceiling:        ", sorted(CEILINGS["triage-agent"]))
print("\nNo ceiling was broken. The agent still reached artifact:write and")
print("deploy:staging, because it can ask something else to do the work.")

## 4 · The control

The fix has two halves and you need both:

**Narrowing on delegation.** When one identity acts for another, the resulting authority must be the *intersection* of what was presented and what the new actor may hold — never the union. This is the mechanism A2.5 builds properly as RFC 8693 token exchange.

**Recording the chain.** The resource server must be able to see that the call arrived through `triage-agent`, so a policy can refuse it even when the immediate caller is allowed.

In [ ]:
def delegate(presented_scopes, presenting, new_actor):
    """Intersection, not union. This one line is the whole control."""
    ceiling = CEILINGS.get(new_actor, set())
    return set(presented_scopes) & ceiling

start = grant("triage-agent", {"repo:read", "finding:comment"})
print("triage-agent holds:      ", sorted(start))
hop1 = delegate(start, "triage-agent", "ci-builder")
print("→ delegated to ci-builder:", sorted(hop1) or "∅ — nothing survives")
hop2 = delegate(hop1, "ci-builder", "deploy-bot")
print("→ delegated to deploy-bot:", sorted(hop2) or "∅ — nothing survives")

print("\nAuthority can only shrink along a chain. The walk-around is closed,")
print("and no reviewer had to notice anything.")

In [ ]:
# Verify: property-test it. Delegation must NEVER widen, for any input.
import itertools, random
random.seed(7)
ids = list(CEILINGS)
violations = 0
for _ in range(2000):
    a, b = random.sample(ids, 2)
    held = set(random.sample(sorted(CEILINGS[a]), k=random.randint(0, len(CEILINGS[a]))))
    out = delegate(held, a, b)
    if not out <= held or not out <= CEILINGS[b]:
        violations += 1
print(f"2000 random delegations, widening violations: {violations}")
assert violations == 0
print("Property holds: result ⊆ presented AND result ⊆ new actor's ceiling.")

## What you just proved

Two grants succeed; three are refused with the ceiling that refused them. The chain demo shows `triage-agent` reaching `artifact:write` and `deploy:staging` without breaking any ceiling. Intersection-based delegation reduces the chain to the empty set, and the 2000-case property test reports zero widening violations.

## Your turn

Find one identity in your estate whose ceiling is effectively "everything" — a break-glass role, a CI admin token. A ceiling cannot constrain it, so its controls have to be time and audit instead. A2.8 builds that.

---

**Next → [A1.4 · Blast radius as a design metric](https://spbreed.github.io/cyber-commons/lessons/A1.4.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.3.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.3.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*